# Ordered Logistic Regression Results: FAIRⁿ Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and visualize the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the `mlcroissant` library and referencing all elements by their `@id` as per the Croissant schema.

### Dataset Source
The dataset Croissant schema is available at the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic dataset info as description, title, and citation
print(f"Dataset name: {getattr(metadata, 'name', '<no name>')}")
print(f"Description: {getattr(metadata, 'description', '<no desc>')}")
print(f"Cite as: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', '<no citation>'))}")


## 2. Data Overview
Review available record sets, fields, and their `@id` attributes.

In [ ]:
# List all record sets by their @id
print("\n--- Record sets available in the dataset ---")
record_sets = dataset.record_sets

record_set_ids = []
for rs in record_sets:
    print(f"@id: {rs.id} | Name: {rs.name}")
    record_set_ids.append(rs.id)

# For each record set, list their fields (columns) using `@id`
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs.id} | Fields:")
    for field in rs.fields:
        print(f"  - Field name: {field.name}, @id: {field.id}, Data type: {field.data_type}")


## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.
Reference entities by their `@id`.


In [ ]:
dataframes = {}
for rs_id in record_set_ids:
    # Load records for this record set by @id
    print(f"\nExtracting data for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Fields for {rs_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  [No records found]")
# For the rest of the notebook pick the first non-empty record set:
main_record_set_id = next(iter(dataframes.keys())) if len(dataframes) else None


## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, and grouping. All references use field `@id`s.

In [ ]:
if main_record_set_id is None:
    print("No dataframes loaded from record sets.")
else:
    df = dataframes[main_record_set_id]
    print(f"\nMain record set @id used for EDA: {main_record_set_id}")
    numeric_columns = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    print(f"Numeric columns available: {numeric_columns}")
    if not numeric_columns:
        print("No numeric fields to perform EDA.")
    else:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fci' else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} (field @id) > {threshold:.2f}:")
        print(filtered_df.head(3))

        # Normalize the numeric field (z-score)
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized field '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head(3))

        # Try to group by a likely categorical field
        possible_group_fields = [col for col in df.columns if df[col].nunique() > 1 and df[col].nunique() < len(df) // 3 and col != numeric_field]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"\nMean of '{numeric_field}' grouped by '{group_field}' (field @id):")
            print(grouped_df.head())
        else:
            print("No suitable field for grouping found.")


## 5. Visualization
Visualize numeric field distributions from the main record set using `matplotlib`/`seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_columns:
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} (Field @id: {numeric_field})")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouped data exists, plot group means for top 10 groups
    if 'grouped_df' in locals():
        grouped_df.sort_values(f"mean_{numeric_field}", ascending=False).head(10).plot(kind="bar")
        plt.title(f"Mean {numeric_field} by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to programmatically access and analyze a FAIR^2 dataset using the `mlcroissant` library:

* All exploration referenced Croissant entities using their `@id` for full traceability.
* Metadata and record sets were loaded, and example summaries, filtering, normalization, and grouping were demonstrated.
* Visualizations were created of numeric fields, and grouping by categorical variables (when available) was shown.

Continue exploring the dataset, referencing the Croissant schema and `@id`s for reproducible and standards-compliant data workflows.